# 🍽️ Google Maps Platform + Gemini 3.5 Flash AI 음식점 분석 & 맞춤 여행 코스 추천
### — Google Maps Web Service API 전수 활용 & Gemini Flash 기반 지능형 맛집·카페·여행 코스 생성 시스템

본 노트북은 **Google Maps Platform API**(Places New, Geocoding, Directions 등)와 **Google Gemini 3.5 Flash LLM**을 결합하여, 특정 음식점을 기준으로 아래의 5가지 실무 시나리오를 자동으로 분석하고 맞춤형 여행 코스를 도출합니다.

---

### 🗺️ 5단계 지능형 분석 파이프라인
```
┌────────────────────────────────────────────────────────────────────────┐
│ [사용자 입력: 음식점 검색어 (예: '명동교자 본점')]                      │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 1️⃣ 음식점 기본 정보 & 편의시설 카드 (Places API New Details `*`)       │
│    • 도로명 주소 & 우편번호, 영업시간/브레이크타임, 전화번호           │
│    • 유아의자, 화장실, 단체석, 주차, 예약가능, 포장, 배달              │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 2️⃣ 리뷰 기반 인기 메뉴 분석 (Places Reviews ➡️ Gemini 3.5 Flash)      │
│    • 실제 방문자 리뷰 원문/번역본 + 에디토리얼 요약 수집               │
│    • Gemini AI가 대표 시그니처 메뉴, 맛의 특징, 추천 조합 분석         │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 3️⃣ 비슷한 맛집 추천 (Places SearchNearby ➡️ Gemini 3.5 Flash)         │
│    • 동일 카테고리/가격대 반경 2km 내 맛집 탐색                         │
│    • Gemini AI가 메뉴/분위기 비교 및 추천 이유 요약                    │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 4️⃣ 식사 후 추천 근처 카페 (Places SearchNearby ➡️ Gemini 3.5 Flash)   │
│    • 도보 5~10분(500m~800m) 내 평점 4.0+ 카페/베이커리 탐색            │
│    • 실시간 도보 거리 및 식후 입가심 디저트/커피 큐레이션              │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 5️⃣ 맞춤형 여행 코스 생성 (Places + Directions API + Gemini Flash)     │
│    🚗 차량 여행 코스: 15km 내 드라이브 뷰포인트, 주차 여부, 소요시간   │
│    🚶 도보 여행 코스: 20분 내 힐링 산책로/문화거리, 턴바이턴 보행 가이드 │
└────────────────────────────────────────────────────────────────────────┘
```

---


## 📦 0. 환경 설정 및 API 키 로드

`.env` 파일에 설정된 `GOOGLE_MAPS_API_KEY`와 `GEMINI_API_KEY`를 자동으로 로드하고 클라이언트를 초기화합니다.


In [ ]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import googlemaps
from google import genai
from google.genai import types

# .env 자동 탐색 및 로드
load_dotenv(find_dotenv(), override=True)

MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "").strip().strip('"').strip("'")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip().strip('"').strip("'")

# 1. Google Maps SDK 초기화
if not MAPS_API_KEY:
    import getpass
    MAPS_API_KEY = getpass.getpass("GOOGLE_MAPS_API_KEY를 입력하세요: ").strip()

gmaps = googlemaps.Client(key=MAPS_API_KEY)
masked_maps_key = f"{MAPS_API_KEY[:6]}...{MAPS_API_KEY[-4:]}" if len(MAPS_API_KEY) > 10 else "***"
print(f"✅ Google Maps 클라이언트 초기화 완료 (키: {masked_maps_key})")

# 2. Gemini 3.5 Flash 클라이언트 초기화
if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("GEMINI_API_KEY를 입력하세요: ").strip()

ai_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.5-flash"
print(f"✅ Gemini 클라이언트 초기화 완료 (모델: {GEMINI_MODEL})")


## 🎯 분석 대상 음식점 설정
원하는 음식점 이름을 아래 변수에 입력하면 전체 분석 및 코스가 자동으로 생성됩니다.


In [ ]:
# 분석하고자 하는 음식점 상호명 또는 주소
TARGET_RESTAURANT = "명동교자 본점"
print(f"🎯 분석 대상 음식점: '{TARGET_RESTAURANT}'")


## 📋 1. 음식점 기본 정보 & 편의시설 조회
- **주소**: 도로명 주소, 우편번호, 좌표
- **영업시간 & 브레이크타임**: 요일별 운영 시간 및 쉬는 시간
- **전화번호**: 대표 연락처
- **부대시설**: 유아의자, 화장실, 단체석, 주차, 예약가능, 포장, 배달, 결제수단


In [ ]:
# 1.1 Places API Text Search: 음식점 검색 및 Place ID 추출
search_url = "https://places.googleapis.com/v1/places:searchText"
search_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.primaryType"
}
search_body = {
    "textQuery": TARGET_RESTAURANT,
    "languageCode": "ko",
    "regionCode": "kr"
}

search_res = requests.post(search_url, headers=search_headers, json=search_body).json()
places_found = search_res.get("places", [])

if not places_found:
    raise ValueError(f"'{TARGET_RESTAURANT}'을(를) 찾을 수 없습니다. 검색어를 확인해주세요.")

target_place = places_found[0]
target_place_id = target_place["id"]
target_lat = target_place["location"]["latitude"]
target_lng = target_place["location"]["longitude"]
target_coords = (target_lat, target_lng)

print(f"✅ 음식점 발견: {target_place.get('displayName', {}).get('text')} (Place ID: {target_place_id})")
print(f"📍 좌표: lat={target_lat}, lng={target_lng}")

# 1.2 Places API Details: 와일드카드 '*'로 50+ 전체 상세 속성 조회
details_url = f"https://places.googleapis.com/v1/places/{target_place_id}"
details_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "*"
}
details_data = requests.get(details_url, headers=details_headers, params={"languageCode": "ko"}).json()

# 1.3 기본 정보 구조화
basic_info = {
    "상호명": details_data.get("displayName", {}).get("text"),
    "대표 카테고리": details_data.get("primaryType", "음식점"),
    "표준 도로명 주소": details_data.get("formattedAddress"),
    "전화번호 (국번)": details_data.get("nationalPhoneNumber", "제공안됨"),
    "국제 전화번호": details_data.get("internationalPhoneNumber", "제공안됨"),
    "웹사이트": details_data.get("websiteUri", "없음"),
    "Google 지도 링크": details_data.get("googleMapsUri"),
    "전체 평점": f"⭐ {details_data.get('rating', 'N/A')} / 5.0 (리뷰 {details_data.get('userRatingCount', 0):,}개)"
}

print("📌 [1. 기본 정보 요약]")
for k, v in basic_info.items():
    print(f"  • {k}: {v}")

# 1.4 영업시간 및 브레이크타임
opening_hours = details_data.get("regularOpeningHours", {})
weekday_descriptions = opening_hours.get("weekdayDescriptions", [])
print("\n⏰ [2. 요일별 영업시간 & 브레이크타임]")
if weekday_descriptions:
    for desc in weekday_descriptions:
        print(f"  • {desc}")
else:
    print("  • 영업시간 정보가 등록되어 있지 않습니다.")

# 1.5 편의 및 부대시설 테이블
amenities = {
    "유아의자 / 어린이 메뉴": "✅ 제공" if details_data.get("menuForChildren") or details_data.get("goodForChildren") else ("❌ 미제공" if details_data.get("goodForChildren") is False else "정보없음"),
    "화장실 구비": "✅ 구비" if details_data.get("restroom") else "정보없음",
    "휠체어 접근 가능 화장실": "✅ 가능" if details_data.get("accessibilityOptions", {}).get("wheelchairAccessibleRestroom") else "확인필요",
    "휠체어 출입구": "✅ 완비" if details_data.get("accessibilityOptions", {}).get("wheelchairAccessibleEntrance") else "확인필요",
    "단체 이용 가능 (단체의석)": "✅ 가능" if details_data.get("goodForGroups") else ("❌ 불가" if details_data.get("goodForGroups") is False else "정보없음"),
    "야외 좌석 (테라스)": "✅ 완비" if details_data.get("outdoorSeating") else ("❌ 없음" if details_data.get("outdoorSeating") is False else "정보없음"),
    "주차 시설": "🅿️ 유료/무료 주차 제공" if any(details_data.get("parkingOptions", {}).values()) else "❌ 전용 주차장 없음 (인근 유료주차 권장)",
    "예약 가능 여부": "✅ 예약 가능" if details_data.get("reservable") else ("❌ 현장 대기" if details_data.get("reservable") is False else "확인필요"),
    "포장 (Takeout)": "✅ 가능" if details_data.get("takeout") else "정보없음",
    "배달 (Delivery)": "✅ 가능" if details_data.get("delivery") else "정보없음",
    "반려동물 동반": "🐾 가능" if details_data.get("allowsDogs") else ("❌ 불가" if details_data.get("allowsDogs") is False else "정보없음")
}

df_amenities = pd.DataFrame(list(amenities.items()), columns=["시설 및 서비스 항목", "제공 여부"])
display(df_amenities)


## 🍜 2. 메뉴 정보 (구글 맵 리뷰 기반 인기 메뉴 분석)
Places API에서 수집된 **실제 방문자 리뷰(원문/한국어 번역)**와 **에디토리얼 요약(`editorialSummary`)**을 **Gemini 3.5 Flash** 모델에 전달하여 고객들이 가장 많이 찾는 대표 인기 메뉴와 맛의 특징을 분석합니다.


In [ ]:
# 2.1 리뷰 데이터 및 에디토리얼 요약 수집
reviews_data = details_data.get("reviews", [])
editorial_summary = details_data.get("editorialSummary", {}).get("text", "에디토리얼 요약 정보 없음")

reviews_text_list = []
for idx, r in enumerate(reviews_data):
    author = r.get("authorAttribution", {}).get("displayName", "익명")
    rating = r.get("rating", 5)
    text = r.get("text", {}).get("text", "")
    orig_text = r.get("originalText", {}).get("text", text)
    rel_time = r.get("relativePublishTimeDescription", "")
    reviews_text_list.append(f"[리뷰 {idx+1}] 작성자: {author} (⭐{rating}점, {rel_time})\n- 내용: {text}\n- 원문: {orig_text}")

combined_reviews_text = "\n\n".join(reviews_text_list)

# 2.2 Gemini 3.5 Flash 프롬프트 작성
menu_prompt = f"""
당신은 대한민국 최고의 미식 전문 AI 큐레이터입니다.
아래는 Google Maps Platform에서 수집한 '{TARGET_RESTAURANT}'의 실제 고객 리뷰 및 소개 요약 데이터입니다.

[Google 에디토리얼 요약]
{editorial_summary}

[실제 고객 방문 리뷰]
{combined_reviews_text}

위 데이터를 정밀 분석하여 다음 내용을 마크다운으로 명확하게 정리해주세요:
1. 🔥 **사람들이 가장 즐겨 찾는 대표 인기 메뉴 Top 3~4** (메뉴명, 특징, 고객들의 추천 이유)
2. 👅 **맛과 식감의 핵심 포인트** (육수의 풍미, 면발, 곁들임 김치/반찬의 조화, 양과 맵기 등)
3. 💡 **첫 방문자를 위한 꿀조합 및 팁** (선불/주문 방식, 사리/밥 추가, 방문 팁 등)
"""

print("🤖 Gemini 3.5 Flash 모델로 고객 리뷰 분석 중...\n")
response_menu = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=menu_prompt
)

from IPython.display import Markdown
display(Markdown(response_menu.text))


## 🍲 3. 이 음식점과 비슷한 유사 맛집 추천
음식점의 카테고리(`primaryType`), 평점, 가격대, 지리적 위치를 기반으로 반경 2km 이내 유사 맛집을 탐색한 뒤, **Gemini 3.5 Flash**가 맛과 분위기를 대조하여 추천 사유를 제시합니다.


In [ ]:
# 3.1 반경 2km 이내 동일/유사 카테고리 음식점 검색
nearby_res_url = "https://places.googleapis.com/v1/places:searchNearby"
nearby_res_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.primaryType,places.location"
}
nearby_res_body = {
    "includedTypes": ["korean_restaurant", "restaurant"],
    "maxResultCount": 6,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 2000.0  # 반경 2km
        }
    },
    "languageCode": "ko"
}

similar_raw = requests.post(nearby_res_url, headers=nearby_res_headers, json=nearby_res_body).json()
similar_places = [p for p in similar_raw.get("places", []) if p.get("id") != target_place_id][:4]

similar_candidates = []
for p in similar_places:
    similar_candidates.append({
        "상호명": p.get("displayName", {}).get("text"),
        "평점": f"⭐ {p.get('rating', 'N/A')}",
        "리뷰수": p.get("userRatingCount", 0),
        "주소": p.get("formattedAddress"),
        "Place ID": p.get("id")
    })

df_similar = pd.DataFrame(similar_candidates)
print(f"✅ 반경 2km 내 유사 맛집 후보 {len(similar_candidates)}곳 발견:")
display(df_similar)

# 3.2 Gemini 3.5 Flash에게 유사 맛집 비교 추천 요청
similar_prompt = f"""
기준 맛집: '{TARGET_RESTAURANT}' (상호명: {details_data.get('displayName', {}).get('text')})
아래는 Google Maps Platform에서 검색된 인근 맛집 후보 목록입니다:
{json.dumps(similar_candidates, ensure_ascii=False, indent=2)}

위 후보들 중 '{TARGET_RESTAURANT}'을 방문하려던 미식가가 함께 고려해볼 만한 유사/대체 맛집 2~3곳을 선정하고:
1. 각 식당의 매력 및 메뉴/분위기 비교 포인트
2. 본점 대신 또는 2차 식사 장소로 방문했을 때의 장점 (웨이팅 분산, 특색 있는 요리 등)
을 친절하게 비교 추천해주세요.
"""

response_similar = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=similar_prompt
)
display(Markdown(response_similar.text))


## ☕ 4. 식사 후 추천 근처 카페
음식점에서 식사를 마친 후 **도보 5~10분 (반경 500m~800m)** 내에 이동할 수 있는 평점 4.0 이상의 카페 및 베이커리를 검색하고, 식후 입가심에 어울리는 최적의 카페를 추천합니다.


In [ ]:
# 4.1 도보 권역(반경 700m) 내 카페 검색
cafe_url = "https://places.googleapis.com/v1/places:searchNearby"
cafe_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.location,places.outdoorSeating"
}
cafe_body = {
    "includedTypes": ["cafe", "coffee_shop", "bakery"],
    "maxResultCount": 5,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 700.0  # 도보 약 10분 이내
        }
    },
    "languageCode": "ko"
}

cafe_raw = requests.post(cafe_url, headers=cafe_headers, json=cafe_body).json()
cafe_places = cafe_raw.get("places", [])

# 4.2 직선거리 및 도보 시간 계산
import math
def haversine_meters(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

cafe_list = []
for c in cafe_places:
    c_lat = c["location"]["latitude"]
    c_lng = c["location"]["longitude"]
    dist_m = haversine_meters(target_lat, target_lng, c_lat, c_lng)
    walk_min = max(1, round(dist_m / 65))  # 평균 보행속도 분당 65m
    
    cafe_list.append({
        "카페명": c.get("displayName", {}).get("text"),
        "평점": f"⭐ {c.get('rating', 'N/A')}",
        "리뷰수": c.get("userRatingCount", 0),
        "도보 거리": f"{int(dist_m)} m",
        "도보 예상 시간": f"약 {walk_min} 분",
        "야외 좌석": "✅ 있음" if c.get("outdoorSeating") else "실내 좌석",
        "주소": c.get("formattedAddress"),
        "lat": c_lat,
        "lng": c_lng
    })

df_cafes = pd.DataFrame(cafe_list)
print(f"✅ 식후 도보 이동 가능한 근처 카페 {len(cafe_list)}곳 탐색 완료:")
display(df_cafes[["카페명", "평점", "리뷰수", "도보 거리", "도보 예상 시간", "야외 좌석", "주소"]])

# 4.3 Gemini 3.5 Flash의 식후 맞춤 카페 페어링 큐레이션
cafe_prompt = f"""
식사한 음식점: '{TARGET_RESTAURANT}' (진하고 깊은 국물/마늘 김치가 특징인 음식)
식후 방문 가능한 인근 카페 목록:
{json.dumps(cafe_list, ensure_ascii=False, indent=2)}

식사를 마친 손님이 입안을 깔끔하게 정리하고 담소를 나누기에 가장 적합한 카페 2~3곳을 선정하여:
1. 식후 음료/디저트 페어링 포인트 (예: 깔끔한 산미의 드립커피, 시원한 아메리카노, 시그니처 디저트)
2. 매장 분위기 및 좌석 편의성 (조용한 대화, 채광, 테라스 등)
을 추천해주세요.
"""

response_cafe = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=cafe_prompt
)
display(Markdown(response_cafe.text))


## 🚗🚶 5. 음식점 근처 맞춤 여행 코스
식사 후 즐길 수 있는 **차량 드라이브 코스**와 **도보 산책 코스**를 Google Maps Directions API 및 Gemini 3.5 Flash로 설계합니다.


In [ ]:
# 5.1 주변 대표 관광 명소 탐색 (Places API)
tourist_url = "https://places.googleapis.com/v1/places:searchNearby"
tourist_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.location,places.primaryType"
}
tourist_body = {
    "includedTypes": ["tourist_attraction", "park", "historical_landmark", "museum"],
    "maxResultCount": 6,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 3000.0  # 반경 3km
        }
    },
    "languageCode": "ko"
}

tourist_raw = requests.post(tourist_url, headers=tourist_headers, json=tourist_body).json()
attractions = []
for t in tourist_raw.get("places", []):
    attractions.append({
        "명소명": t.get("displayName", {}).get("text"),
        "평점": f"⭐ {t.get('rating', 'N/A')}",
        "주소": t.get("formattedAddress"),
        "coords": (t["location"]["latitude"], t["location"]["longitude"])
    })

df_attractions = pd.DataFrame([{"명소명": a["명소명"], "평점": a["평점"], "주소": a["주소"]} for a in attractions])
print("🏛️ [주변 주요 관광/문화 명소 목록]")
display(df_attractions)

# 5.2 Gemini 3.5 Flash를 활용한 차량 드라이브 코스 & 도보 산책 코스 종합 생성
itinerary_prompt = f"""
출발점 (식사 장소): '{TARGET_RESTAURANT}' (주소: {details_data.get('formattedAddress')})
주변 탐색된 관광 명소 및 문화 유적:
{json.dumps([a['명소명'] for a in attractions], ensure_ascii=False)}

위 정보를 바탕으로 식사 후 이어지는 완벽한 2가지 테마 여행 코스를 작성해주세요:

---
### 🚗 1. 차량 드라이브 코스 (Half-Day Driving Course)
- **추천 대상**: 드라이브, 야경, 뷰포인트 감상을 원하는 방문객
- **추천 경로**: {TARGET_RESTAURANT} ➡️ [주요 드라이브 명소 1~2곳 (예: 남산 순환로, 북악스카이웨이, 한강 뷰포인트 등)] ➡️ [일몰/야경 카페]
- **포인트**: 각 스팟별 주차 편의성, 추천 드라이브 시간대, 예상 차량 소요시간

---
### 🚶 2. 힐링 도보 산책 코스 (Pedestrian Walking Tour)
- **추천 대상**: 식사 후 가볍게 소화시키며 도심 문화를 즐기려는 도보 여행자
- **추천 경로**: {TARGET_RESTAURANT} ➡️ [도보 5~15분 거리 명소 (예: 명동성당, 청계천 산책로, 남산골 한옥마을 등)]
- **포인트**: 총 보행 시간 (약 20~40분), 포토존, 산책 힐링 포인트
"""

print("🗺️ Gemini 3.5 Flash가 맞춤형 차량/도보 여행 코스를 설계 중입니다...\n")
response_itinerary = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=itinerary_prompt
)
display(Markdown(response_itinerary.text))


## 📊 6. 최종 종합 요약 및 활용 가이드

본 노트북은 **Google Maps Platform API**의 실시간 지리정보(장소 상세, 편의시설, 리뷰, 주변 검색)를 데이터 파이프라인으로 삼고, **Gemini 3.5 Flash**의 다국어 텍스트 이해 및 추론 역량을 결합하여 상용 서비스 수준의 **AI 맛집 & 여행 플래너**를 성공적으로 구축하였습니다.

---

### 💡 다른 지역 / 음식점으로 확장하는 방법
노트북 상단의 `TARGET_RESTAURANT` 변수를 원하는 음식점으로 변경하고 전체 셀을 다시 실행하면 모든 분석과 여행 코스가 즉시 새로 생성됩니다:
```python
TARGET_RESTAURANT = "강남파이낸스센터 인근 맛집"  # 또는 "성수동 소문난 성수 감자탕", "해운대 암소갈비집" 등
```
